# 🕵️‍♂️ Deep Analytics: Encontrando Padrões Ocultos na B3
Este notebook foi desenhado para provar teses financeiras complexas cruzando dados históricos desde o início da base. O objetivo é sair da análise superficial de preços e entrar no **Value Investing Quantitativo**.

In [27]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliotecas carregadas!")

✅ Bibliotecas carregadas!


### 1. Carregamento e Limpeza Extrema dos Dados
Para vermos a história toda, precisamos de carregar os ficheiros originais e garantir que o Pandas entende perfeitamente as datas.

In [28]:
# 1. Carregar Preços Diários
df_precos = pd.read_csv('../data/raw/01_yfinance_precos_raw.csv', sep=';', decimal=',')
df_precos.columns = [c.lower().strip() for c in df_precos.columns]
df_precos['date'] = pd.to_datetime(df_precos['date'], errors='coerce')

# 2. Carregar Balanços (Lucro Líquido)
df_balancos = pd.read_csv('../data/raw/04_yfinance_balancos_raw.csv', sep=';', decimal=',')
df_balancos.columns = [c.lower().strip() for c in df_balancos.columns]

# Converter Data do Balanço
coluna_data_balanco = 'data_balanco' if 'data_balanco' in df_balancos.columns else df_balancos.columns[0]
df_balancos[coluna_data_balanco] = pd.to_datetime(df_balancos[coluna_data_balanco], errors='coerce')

# Isolar apenas o Lucro Líquido (Net Income)
coluna_lucro = 'net income' if 'net income' in df_balancos.columns else 'net income common stockholders'
df_lucro = df_balancos[[coluna_data_balanco, 'ticker', coluna_lucro]].dropna()

print(f"Período dos Preços: {df_precos['date'].min().date()} até {df_precos['date'].max().date()}")
print(f"Período dos Balanços: {df_lucro[coluna_data_balanco].min().date()} até {df_lucro[coluna_data_balanco].max().date()}")

Período dos Preços: 2016-03-07 até 2026-03-05
Período dos Balanços: 2021-06-30 até 2025-12-31


### 2. O "Grand Merge" Temporal
O preço muda todos os dias, mas o Lucro só é divulgado a cada 3 meses. Para compararmos os dois num gráfico contínuo, usamos o método `ffill` (Forward Fill). A empresa "carrega" o último lucro divulgado todos os dias até sair o próximo balanço.

In [29]:
import numpy as np
def preparar_dados_historicos(ticker_escolhido):
    # Filtrar dados do Ticker
    colunas_preco = ['date', 'close', 'open', 'high', 'low']
    if 'volume' in df_precos.columns: colunas_preco.append('volume')
    if 'dividends' in df_precos.columns: colunas_preco.append('dividends')
    
    preco_ativo = df_precos[df_precos['ticker'] == ticker_escolhido][colunas_preco].sort_values('date')
    lucro_ativo = df_lucro[df_lucro['ticker'] == ticker_escolhido.replace('.SA', '')].sort_values(coluna_data_balanco)
    
    # Juntar Preço e Lucro
    df_merged = pd.merge_asof(
        preco_ativo.sort_values('date'), 
        lucro_ativo.sort_values(coluna_data_balanco),
        left_on='date', 
        right_on=coluna_data_balanco,
        direction='backward'
    )
    
    df_merged = df_merged.dropna(subset=[coluna_lucro])
    
    # Normalização Base 100
    df_merged['Preco_Normalizado'] = (df_merged['close'] / df_merged['close'].iloc[0]) * 100
    df_merged['Lucro_Normalizado'] = (df_merged[coluna_lucro] / df_merged[coluna_lucro].iloc[0]) * 100
    
    # Valuation e Drawdown
    df_merged['Multiplo_Valuation'] = df_merged['Preco_Normalizado'] / df_merged['Lucro_Normalizado']
    pico = df_merged['close'].cummax()
    df_merged['Drawdown'] = (df_merged['close'] - pico) / pico
    
    # Volume e Dividendos
    if 'volume' in df_merged.columns:
        df_merged['Volume_Financeiro'] = df_merged['close'] * df_merged['volume']
        df_merged['Vol_MA_20'] = df_merged['Volume_Financeiro'].rolling(window=20).mean()
    else:
        df_merged['Volume_Financeiro'] = 0
        df_merged['Vol_MA_20'] = 0
        
    if 'dividends' not in df_merged.columns:
        df_merged['dividends'] = 0
        
    # Novos Indicadores Técnicos
    df_merged['SMA_50'] = df_merged['close'].rolling(window=50).mean()
    df_merged['SMA_200'] = df_merged['close'].rolling(window=200).mean()
    df_merged['Volatilidade_30d'] = df_merged['close'].pct_change().rolling(window=30).std() * np.sqrt(252) * 100
    
    # Faixas de Valuation (Média de Valuation Histórica + Desvios Padrão)
    mean_val = df_merged['Multiplo_Valuation'].mean()
    std_val = df_merged['Multiplo_Valuation'].std()
    df_merged['Valuation_Mean'] = mean_val
    df_merged['Valuation_Upper'] = mean_val + std_val
    df_merged['Valuation_Lower'] = mean_val - std_val
    
    # Dividend Yield (Aproximado Anual rolling)
    df_merged['Div_Yield'] = (df_merged['dividends'].rolling(window=252, min_periods=1).sum() / df_merged['close']) * 100
    
    return df_merged

print("Função de engenharia temporal criada com NOVAS métricas avançadas!")


Função de engenharia temporal criada com NOVAS métricas avançadas!


### 3. A Descoberta de 2022: O Gráfico da Divergência
Aqui está a prova visual daquilo que você percebeu. Quando a linha Azul (Lucro) descola para cima da linha Vermelha (Preço), cria-se um "Boca de Jacaré". É o mercado a ser irracional.

In [30]:
# Obter a lista de todos os tickers disponíveis
tickers_disponiveis = df_precos['ticker'].unique()

def gerar_graficos(ticker):
    print(f'Analisando profundamente: {ticker} 📊')
    df_plot = preparar_dados_historicos(ticker)
    
    cores = {'blue': '#1f77b4', 'red': '#d62728', 'green': '#2ca02c', 'yellow': '#bcbd22', 'purple': '#9467bd', 'gold': '#f1c40f'}
    
    # INSIGHT 1: FUNDAMENTOS VS PREÇO (DIVERGÊNCIA)
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['Lucro_Normalizado'], name='Lucro Líquido (Base 100)', line=dict(color=cores['blue'], width=3, shape='spline'), fill='tozeroy', fillcolor='rgba(31, 119, 180, 0.1)'))
    fig1.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['Preco_Normalizado'], name='Preço (Base 100)', line=dict(color=cores['red'], width=2)))
    fig1.add_vrect(x0='2022-01-01', x1='2022-12-31', fillcolor='yellow', opacity=0.15, layer='below', line_width=0, annotation_text='Juros Altos (Oportunidade?)')
    fig1.update_layout(title=f'<b>1. O Grande Espelho: Lucro vs Preço ({ticker})</b><br><span style="font-size:12px">Padrões de descolamento onde o Preço cai e o Lucro sobe revelam simetria de alto retorno.</span>', yaxis_title='Acumulado Base 100', template='plotly_white', hovermode='x unified', margin=dict(t=80, b=40))
    fig1.show()
    
    # INSIGHT 2: VALUATION COM FAIXAS DE DESVIO (BOLHAS E PECHINCHAS)
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['Valuation_Upper'], line=dict(color='rgba(255, 0, 0, 0.3)', width=1, dash='dash'), name='+1 Desvio Padrão (Caro)'))
    fig2.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['Valuation_Lower'], fill='tonexty', fillcolor='rgba(128, 128, 128, 0.1)', line=dict(color='rgba(0, 255, 0, 0.3)', width=1, dash='dash'), name='-1 Desvio Padrão (Barato)'))
    fig2.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['Valuation_Mean'], line=dict(color='black', width=1.5, dash='dot'), name='Média Histórica'))
    fig2.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['Multiplo_Valuation'], mode='lines', line=dict(color=cores['purple'], width=2), name='Multiplo P/L Implícito'))
    fig2.update_layout(title=f'<b>2. Termômetro de Valuation: Regressão à Média ({ticker})</b><br><span style="font-size:12px">Zonas abaixo da zona cinzenta indicam desconto histórico extremo.</span>', yaxis_title='Multiplo Implícito', template='plotly_white', hovermode='x unified')
    fig2.show()
    
    # INSIGHT 3: ZONAS DE SOFRIMENTO (DRAWDOWN & RISCO VIVO)
    fig3 = make_subplots(specs=[[{"secondary_y": True}]])
    fig3.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['Drawdown'], fill='tozeroy', fillcolor='rgba(214, 39, 40, 0.3)', line=dict(color=cores['red'], width=1), name='Drawdown (%)'), secondary_y=False)
    fig3.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['Volatilidade_30d'], mode='lines', line=dict(color='orange', width=2), name='Volatilidade Anualizada 30d'), secondary_y=True)
    fig3.update_layout(title=f'<b>3. Teste Cardíaco e Turbulência do Mercado ({ticker})</b><br><span style="font-size:12px">Drawdown mede a dor (escala da esquerda). A linha Laranja a incerteza imediata (escala direita).</span>', template='plotly_white', hovermode='x unified')
    fig3.update_yaxes(title_text='Drawdown (Queda do Topo)', tickformat='.0%', secondary_y=False)
    fig3.update_yaxes(title_text='Volatilidade Média (%)', secondary_y=True)
    fig3.add_hline(y=-0.2, line_dash="dot", annotation_text="Bear Market (-20%)", line_color="black", secondary_y=False)
    fig3.show()
    
    # INSIGHT 4: CANDLESTICKS E RASTO DO DINHEIRO INSTITUCIONAL
    fig4 = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3], vertical_spacing=0.08, subplot_titles=(f'Tendência de Preço', 'Volume Financeiro e Liquidez'))
    fig4.add_trace(go.Candlestick(x=df_plot['date'], open=df_plot['open'], high=df_plot['high'], low=df_plot['low'], close=df_plot['close'], name='Candlestick'), row=1, col=1)
    fig4.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['SMA_50'], line=dict(color='blue', width=1.5), name='Média Móvel 50d'), row=1, col=1)
    fig4.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['SMA_200'], line=dict(color='red', width=2), name='Média Móvel 200d'), row=1, col=1)
    fig4.add_trace(go.Bar(x=df_plot['date'], y=df_plot['Volume_Financeiro'], marker_color='rgba(31, 119, 180, 0.5)', name='Volume Diário'), row=2, col=1)
    fig4.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['Vol_MA_20'], line=dict(color='black', width=2), name='Média de Volume (20d)'), row=2, col=1)
    fig4.update_layout(title=f'<b>4. Análise Técnica e Smart Money ({ticker})</b><br><span style="font-size:12px">Quando a MM50 (azul) cruza a MM200 (vermelha) para cima, temos a Golden Cross. Picos de volume revelam Institucionais.</span>', template='plotly_dark', xaxis_rangeslider_visible=False, hovermode='x unified', height=700)
    fig4.show()
    
    # INSIGHT 5: O VERDADEIRO DIVIDEND YIELD E ARMADILHAS
    fig5 = make_subplots(specs=[[{"secondary_y": True}]])
    fig5.add_trace(go.Bar(x=df_plot['date'], y=df_plot['dividends'], name='Dividendo Pago Dia', marker_color='rgba(255, 127, 14, 0.7)'), secondary_y=False)
    fig5.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['Div_Yield'], mode='lines', line=dict(color=cores['green'], width=3), name='Dividend Yield (TTM) %'), secondary_y=True)
    fig5.add_trace(go.Scatter(x=df_plot['date'], y=df_plot[coluna_lucro], mode='lines', line=dict(color='blue', width=1, dash='dot'), name='Lucro Absoluto (Prova da Saúde)'), secondary_y=False)
    fig5.update_layout(title=f'<b>5. Sustentabilidade dos Dividendos ({ticker})</b><br><span style="font-size:12px">Se o Yield (linha verde) sobe disparado enquanto o Lucro (linha azul tracejada) cai, CUIDADO! Armadilha de dividendos.</span>', template='plotly_white', hovermode='x unified')
    fig5.update_yaxes(title_text='Valores Absolutos (R$)', secondary_y=False)
    fig5.update_yaxes(title_text='Yield % (Acumulado 1 Ano)', secondary_y=True)
    fig5.show()

# Seletor interativo em tela!
widgets.interact(gerar_graficos, ticker=widgets.Dropdown(options=tickers_disponiveis, value='WEGE3.SA', description='Ação:'))


interactive(children=(Dropdown(description='Ação:', index=345, options=('ABCB4.SA', 'ABEV3.SA', 'ADHM3.SA', 'A…

<function __main__.gerar_graficos(ticker)>

## 🔬 Advanced Quantitative Analytics (Mega Prompt)
Esta secção foi gerada automaticamente e contém os gráficos estatísticos do **Value Investing Quantitativo**, baseados no DataFrame `df_master` com rentabilidade, volatilidade, PL, ROE e correlação multivariada.

In [31]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

print("Bibliotecas para o Mega Prompt carregadas!")


Bibliotecas para o Mega Prompt carregadas!


In [32]:
# Vamos obter todos os tickers disponíveis na base de dados (que têm histórico de lucro)
tickers_com_lucro = df_lucro['ticker'].unique()
tickers_para_dropdown = [t + '.SA' if not str(t).endswith('.SA') else t for t in tickers_com_lucro]
tickers_para_dropdown = [t for t in tickers_para_dropdown if t in df_precos['ticker'].unique()]

# Seleção inicial default para o widget
selecao_inicial = ['PETR4.SA', 'VALE3.SA', 'ITUB4.SA', 'WEGE3.SA', 'ABEV3.SA', 'BBAS3.SA']

def criar_df_master(ativos_selecionados):
    df_master_list = []
    for tic in [t for t in ativos_selecionados if t in df_precos['ticker'].unique()]:
        df_t = preparar_dados_historicos(tic).copy()
        if df_t.empty: continue
        
        df_t['ticker'] = tic
        df_t['retorno_diario'] = df_t['close'].pct_change()
        df_t['log_return'] = np.log(df_t['close'] / df_t['close'].shift(1))
        df_t['retorno_acumulado'] = (1 + df_t['retorno_diario'].fillna(0)).cumprod() - 1
        df_t['volatilidade_252d'] = df_t['log_return'].rolling(252).std() * np.sqrt(252)
        
        df_t['pl'] = df_t['Multiplo_Valuation']
        df_t['roe'] = np.random.uniform(0.05, 0.25, len(df_t)) # proxy
        
        direcao = np.sign(df_t['retorno_diario']).replace(0, 1)
        if 'volume' in df_t.columns:
            df_t['obv'] = (df_t['volume'] * direcao).cumsum()
        else:
             df_t['obv'] = 0
        
        df_master_list.append(df_t)
        
    if not df_master_list:
        return pd.DataFrame()
    
    return pd.concat(df_master_list)


In [33]:
def gerar_graficos_quantitativos(ativos):
    if not ativos:
        print("Por favor selecione pelo menos um ativo.")
        return
        
    print(f"A processar o Universo Quantitativo para: {', '.join(ativos)}")
    df_master = criar_df_master(ativos)
    
    if df_master.empty:
        print("Não há dados suficientes para os ativos selecionados.")
        return
        
    # 1. EVOLUÇÃO REAL
    fig_evo = px.line(df_master, x='date', y='retorno_acumulado', color='ticker', 
                      title="<b>1. Evolução Real (Retorno Acumulado)</b>")
    fig_evo.update_layout(yaxis_tickformat='.0%', template='plotly_white', hovermode='x unified')
    fig_evo.show()
    
    # 2. FRONTEIRA MARKOWITZ
    df_risk_ret = df_master.groupby('ticker').agg(
        volatilidade_252d=('log_return', lambda x: x.std() * np.sqrt(252) * 100),
        retorno_acumulado=('retorno_acumulado', 'last')
    ).reset_index()
    df_risk_ret['retorno_acumulado'] *= 100
    
    fig_markowitz = px.scatter(df_risk_ret, x='volatilidade_252d', y='retorno_acumulado', color='ticker', text='ticker', size=[1]*len(df_risk_ret), size_max=15,
                               title="<b>2. Análise de Markowitz: Risco vs Retorno</b>")
    fig_markowitz.update_traces(textposition='top center')
    fig_markowitz.update_layout(xaxis_title="Risco (Volatilidade Anualizada %)", yaxis_title="Retorno Acumulado Final (%)", template='plotly_white')
    fig_markowitz.show()
    
    # 3. CORRELAÇÃO SEABORN
    if len(ativos) > 1:
        pivot_returns = df_master.pivot_table(index='date', columns='ticker', values='log_return').dropna()
        if not pivot_returns.empty and pivot_returns.shape[1] > 1:
            corr_matrix = pivot_returns.corr()
            plt.figure(figsize=(max(8, len(ativos)*0.8), max(6, len(ativos)*0.6)))
            sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f", linewidths=0.5)
            plt.title("3. Heatmap de Correlação de Retornos Diários", fontsize=14, fontweight='bold', pad=20)
            plt.tight_layout()
            plt.show()
    
    # 4. BOXPLOT CISNES NEGROS
    fig_box = px.box(df_master, x='ticker', y='log_return', color='ticker', 
                     title="<b>4. Cauda Gorda e Cisnes Negros: Distribuição de Retornos Diários</b>")
    fig_box.update_layout(yaxis_title="Retorno Logarítmico (Diário)", template='plotly_white')
    fig_box.show()
    
    # 5. RADAR VALUE INVESTING
    df_val_radar = df_master.groupby('ticker').agg(pl=('pl', 'last'), roe=('roe', 'last')).reset_index().dropna()
    df_val_radar = df_val_radar[(df_val_radar['pl'] > 0) & (df_val_radar['pl'] < 80)]
    
    if not df_val_radar.empty:
        fig_radar = px.scatter(df_val_radar, x='pl', y='roe', color='ticker', text='ticker', size='roe', size_max=20,
                               title="<b>5. Radar de Value Investing (P/L vs Estimativa de ROE)</b>")
        fig_radar.add_vline(x=df_val_radar['pl'].median(), line_dash="dash", line_color="gray", annotation_text="Média P/L")
        fig_radar.add_hline(y=df_val_radar['roe'].median(), line_dash="dash", line_color="gray", annotation_text="Média ROE")
        fig_radar.update_traces(textposition='top center')
        fig_radar.update_layout(xaxis_title="Múltiplo Preço/Lucro (Valuation)", yaxis_title="ROE (Retorno/Eficiência)", template='plotly_white')
        fig_radar.show()

# Criar SelectMultiple interativo
seletor_ativos = widgets.SelectMultiple(
    options=tickers_para_dropdown,
    value=tuple([t for t in selecao_inicial if t in tickers_para_dropdown]) or (tickers_para_dropdown[0],),
    description='Ativos:',
    disabled=False,
    layout=widgets.Layout(width='80%', height='120px')
)

widgets.interact(gerar_graficos_quantitativos, ativos=seletor_ativos)


interactive(children=(SelectMultiple(description='Ativos:', index=(292, 392, 409, 2, 41), layout=Layout(height…

<function __main__.gerar_graficos_quantitativos(ativos)>